# 06 — Mars process validation

This notebook follows Mars region discovery. Notebook 03 found connected regions, neutral property types, and unusual regions; it did **not** prove that a cluster is volcanism, ice, water alteration, or wind activity.

Notebook 06 connects those neutral outputs to a literature-backed process registry and creates a targeted review queue. A `candidate_only` row is a follow-up target, never a confirmed process. Missing morphology, repeat-change, radar, or high-resolution mineral observations are recorded explicitly.

Mars is handled separately from the Moon. Earth matching remains disabled until processes are independently validated.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None

INPUT_ROOT_OVERRIDE = None  # Set to the folder containing regions/mars_regions.parquet when needed.
OUTPUT_ROOT = Path("/kaggle/working/mars_process_validation")


In [ ]:
def find_discovery_root(override: str | None = None) -> Path:
    required = Path("regions") / "mars_regions.parquet"
    bases = [Path(override)] if override else []
    bases += [Path("/kaggle/working"), Path("/kaggle/input"), Path.cwd()]
    checked = []
    for base in bases:
        if not base.exists():
            continue
        if (base / required).exists():
            return base
        for found in base.glob("**/mars_regions.parquet"):
            candidate = found.parent.parent
            if (candidate / required).exists():
                return candidate
        checked.append(str(base))
    raise FileNotFoundError(
        "Notebook 03 outputs were not found. Attach its saved output folder/dataset or set "
        f"INPUT_ROOT_OVERRIDE. Checked: {checked}"
    )


INPUT_ROOT = find_discovery_root(INPUT_ROOT_OVERRIDE)
REGION_DIR = INPUT_ROOT / "regions"
REPORT_DIR = INPUT_ROOT / "reports"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
regions_df = pd.read_parquet(REGION_DIR / "mars_regions.parquet")
feature_distributions_path = REGION_DIR / "mars_region_feature_distributions.parquet"
neutral_distributions_path = REGION_DIR / "mars_neutral_type_distributions.parquet"
region_feature_distributions_df = pd.read_parquet(feature_distributions_path) if feature_distributions_path.exists() else pd.DataFrame()
neutral_distributions_df = pd.read_parquet(neutral_distributions_path) if neutral_distributions_path.exists() else pd.DataFrame()
reference_path = REPORT_DIR / "reference_locations.csv"
reference_locations_df = pd.read_csv(reference_path) if reference_path.exists() else pd.DataFrame()
print(f"Input: {INPUT_ROOT}")
print(f"Regions: {len(regions_df):,}")
print(f"Neutral types: {regions_df.get('neutral_type', pd.Series(dtype=str)).nunique(dropna=True):,}")


## Measurement inventory

These are measurements available in the region table. The process registry below also asks for evidence that is not in the global products, such as morphology, repeat images, radar, and illumination history. Those missing families are not guessed from cluster values.


In [ ]:
def matching_columns(frame: pd.DataFrame, patterns: tuple[str, ...]) -> list[str]:
    # Presence must come from a physical measurement, not metadata such as
    # missing_neutron, coverage, quality flags, standard deviations, or imputed model values.
    direct_prefixes = ("raw_mean__", "raw_positive_fraction__")
    excluded_tokens = ("missing_", "coverage", "valid_fraction")
    return [
        c for c in frame.columns
        if c.startswith(direct_prefixes)
        and not any(token in c.lower() for token in excluded_tokens)
        and any(pattern in c.lower() for pattern in patterns)
    ]

FAMILY_PATTERNS = {
    "terrain": ("elevation", "slope", "roughness", "relief", "terrain"),
    "thermal": ("thermal", "temperature", "inertia"),
    "albedo": ("albedo", "reflectance"),
    "composition": ("mineral", "spectral", "emissivity", "silica", "olivine", "pyroxene"),
    "hydrogen_sensitive": ("neutron", "hydrogen"),
    "dust": ("dust", "surface_dust"),
    "morphology": ("morphology", "dune", "channel", "crater", "landform"),
    "repeat_change": ("repeat", "change_detection", "migration", "temporal_change"),
    "radar_subsurface": ("sharad", "mini_rf", "radar", "subsurface"),
    "illumination_history": ("illumination", "shadow", "solar", "insolation"),
}
available_measurements = {family: matching_columns(regions_df, patterns) for family, patterns in FAMILY_PATTERNS.items()}
measurement_inventory_df = pd.DataFrame([
    {"measurement_family": family, "available_columns": ", ".join(cols), "available": bool(cols)}
    for family, cols in available_measurements.items()
])
display(measurement_inventory_df)


## Literature-backed process registry

The papers define evidence requirements; they do not label clusters. The registry includes current processes, past processes, and useful environments. It deliberately keeps alternatives and Earth-observable variables visible.


In [ ]:
PROCESS_REGISTRY = [
 {"process_id":"active_aeolian_transport","kind":"current_process","required":["morphology","repeat_change"],"supporting":["thermal","dust","terrain"],"alternatives":"Static dunes may be inactive; repeat displacement must exceed registration uncertainty.","earth":"dune/ripple morphology, repeat displacement, wind and sediment availability","refs":[("Bridges et al. 2012 — Earth-like sand fluxes","https://doi.org/10.1038/nature11022"),("Baker et al. 2018 — Bagnold dune transport","https://doi.org/10.1029/2018GL079040")]},
 {"process_id":"past_aeolian_erosion_or_deposition","kind":"past_process","required":["morphology","terrain"],"supporting":["dust","thermal","albedo"],"alternatives":"Wind-shaped forms can be confused with fluvial, volcanic, or structural relief.","earth":"yardangs, dunes, ventifacts, sediment texture and wind context","refs":[("Greeley & Iversen 1985 — Wind as a geological process","https://doi.org/10.1016/0019-1035(85)90029-6"),("Silvestro et al. 2010 — Dune migration","https://doi.org/10.1016/j.geomorph.2010.02.010")]},
 {"process_id":"past_fluvial_channel_network","kind":"past_process","required":["morphology","terrain"],"supporting":["composition","albedo"],"alternatives":"Lava channels, tectonic troughs, and inverted relief require explicit testing.","earth":"dendritic channels, drainage gradients, valley cross-sections and deposits","refs":[("Fassett & Head 2008 — Valley-network timing","https://doi.org/10.1016/j.icarus.2007.12.009"),("Hynek et al. 2010 — Global valley networks","https://doi.org/10.1016/j.icarus.2009.10.008")]},
 {"process_id":"past_lacustrine_delta_or_basin","kind":"past_process","required":["morphology","terrain"],"supporting":["composition","hydrogen_sensitive"],"alternatives":"Fans and layered deposits need basin geometry and stratigraphic context.","earth":"closed-basin geometry, delta fronts, shoreline levels and sediments","refs":[("Ehlmann et al. 2011 — Clay formation and subsurface water","https://doi.org/10.1038/nature09903"),("Goudge et al. 2018 — Jezero sedimentary rocks","https://doi.org/10.1016/j.icarus.2017.10.024")]},
 {"process_id":"water_driven_mineral_alteration","kind":"past_process","required":["composition"],"supporting":["morphology","terrain","hydrogen_sensitive"],"alternatives":"Minerals alone do not distinguish surface, subsurface, hydrothermal, or transported alteration.","earth":"clay/sulfate/carbonate spectra, host geology, alteration zonation and fluid pathways","refs":[("Bibring et al. 2006 — Aqueous Mars history","https://doi.org/10.1126/science.1122659"),("Ehlmann et al. 2009 — Aqueous mineralogy synthesis","https://doi.org/10.1029/2009JE003342"),("Carter et al. 2013 — Hydrated minerals","https://doi.org/10.1016/j.icarus.2013.03.004")]},
 {"process_id":"volcanic_lava_emplacement","kind":"past_process","required":["morphology","terrain"],"supporting":["composition","thermal","albedo"],"alternatives":"Mafic composition is supporting evidence only; volcanic morphology and context are required.","earth":"flow fronts, vents, lava textures, minerals and age relationships","refs":[("Head et al. 2002 — Recent Martian volcanism","https://doi.org/10.1029/2000JE001445"),("Plescia 2004 — Martian volcano morphology","https://doi.org/10.1016/j.icarus.2003.08.003")]},
 {"process_id":"explosive_or_pyroclastic_volcanism","kind":"past_process","required":["morphology","composition"],"supporting":["thermal","terrain"],"alternatives":"Layered deposits may be sedimentary or aeolian; vents and deposit geometry matter.","earth":"tephra-like mantles, vent proximity, volcanic glass/mineral spectra and stratigraphy","refs":[("Wray et al. 2013 — Ancient explosive volcanism","https://doi.org/10.1016/j.icarus.2012.10.014"),("Kerber et al. 2012 — Tharsis explosive volcanism","https://doi.org/10.1016/j.icarus.2011.12.015")]},
 {"process_id":"glacial_or_periglacial_modification","kind":"past_process_or_environment","required":["morphology","terrain"],"supporting":["hydrogen_sensitive","thermal","radar_subsurface"],"alternatives":"Volcanic, aeolian, and mass-wasting explanations must be tested.","earth":"debris-covered glaciers, moraines, patterned ground and ice content","refs":[("Head et al. 2003 — Recent ice ages","https://doi.org/10.1038/nature02114"),("Head et al. 2005 — Tropical glaciation","https://doi.org/10.1038/nature03359"),("Levy et al. 2010 — Glacial/periglacial landforms","https://doi.org/10.1016/j.icarus.2009.07.036")]},
 {"process_id":"near_surface_ice_or_ground_ice","kind":"environment_or_resource","required":["hydrogen_sensitive"],"supporting":["thermal","radar_subsurface","morphology"],"alternatives":"Low epithermal-neutron flux indicates hydrogen, not uniquely water ice; hydrated minerals, adsorbed water, CO2 frost effects, and spatial mixing remain alternatives.","earth":"permafrost, ice content, radar structure, temperature regime and soil texture","refs":[("Feldman et al. 2002 — Odyssey neutron mapping","https://doi.org/10.1126/science.1073541"),("Feldman et al. 2004 — Near-surface hydrogen","https://doi.org/10.1029/2003JE002160"),("Stuurman et al. 2016 — Buried ice radar evidence","https://doi.org/10.1016/j.icarus.2016.06.013")]},
 {"process_id":"mass_wasting","kind":"current_or_past_process","required":["morphology","terrain"],"supporting":["repeat_change","thermal"],"alternatives":"Slope streaks and deposits can be aeolian or illumination effects.","earth":"slope, scarps, debris aprons, talus, rockfall or landslide change","refs":[("McEwen et al. 2007 — Mars slope streaks","https://doi.org/10.1126/science.1140704"),("Grotzinger et al. 2015 — Gale sedimentary processes","https://doi.org/10.1126/science.aac7575")]},
 {"process_id":"impact_excavation_and_ejecta","kind":"past_process","required":["morphology","terrain"],"supporting":["thermal","albedo","composition"],"alternatives":"Volcanic calderas and collapse pits can mimic crater-like depressions.","earth":"crater/ejecta geometry, blocks, impact melt and shock minerals","refs":[("Kreslavsky & Head 2009 — Crater thermal effects","https://doi.org/10.1016/j.icarus.2008.08.010"),("Robbins & Hynek 2012 — Martian crater database","https://doi.org/10.1016/j.jgr.2011.10.012")]},
 {"process_id":"recent_impact_or_ejecta_change","kind":"recent_process","required":["morphology","repeat_change"],"supporting":["thermal","albedo"],"alternatives":"Seasonal dust or illumination changes can mimic a new disturbance.","earth":"new crater detection, fresh ejecta, seismic/impact monitoring and repeat imagery","refs":[("Dauber et al. 2013 — New Martian impact craters","https://doi.org/10.1016/j.icarus.2013.05.021"),("Robbins et al. 2015 — Mars impact rate","https://doi.org/10.1016/j.icarus.2014.12.036")]},
 {"process_id":"regolith_mantling_or_dust_cover","kind":"environment_or_surface_process","required":["dust","albedo"],"supporting":["thermal","terrain","composition"],"alternatives":"Albedo/thermal differences can reflect rock type, grain size, or illumination.","earth":"dust thickness, grain size, albedo, thermal inertia and atmospheric dust","refs":[("Ruff & Christensen 2002 — Bright/dark regions","https://doi.org/10.1029/2000JE001365"),("Lemmon et al. 2015 — Global dust storm changes","https://doi.org/10.1002/2015GL066353")]},
 {"process_id":"polar_cold_trapping","kind":"environment_or_resource","required":["thermal","illumination_history"],"supporting":["hydrogen_sensitive","radar_subsurface"],"alternatives":"Cold conditions imply preservation potential, not a specific volatile or confirmed deposit.","earth":"temperature history, shadow duration, volatile stability, ice/permafrost and illumination","refs":[("Mellon et al. 2004 — Ground-ice thermal stability","https://doi.org/10.1029/2003JE002064"),("Vasavada et al. 2012 — Mars polar energy balance","https://doi.org/10.1016/j.icarus.2012.02.001")]},
]

literature_registry_df = pd.DataFrame([
 {"process_id": p["process_id"], "kind": p["kind"], "required_families": ", ".join(p["required"]), "supporting_families": ", ".join(p["supporting"]), "alternatives": p["alternatives"], "earth_variables": p["earth"], "references": " | ".join(f"{title} ({url})" for title, url in p["refs"])}
 for p in PROCESS_REGISTRY
])
display(literature_registry_df[["process_id", "kind", "required_families", "supporting_families"]])
print(f"Literature-backed process hypotheses: {len(literature_registry_df)}")


In [ ]:
def family_available(family: str) -> bool:
    return bool(available_measurements.get(family))

# Presence is measured per region. A product existing somewhere in the table
# does not mean it is co-located with every other product.
region_family_presence = regions_df.copy()
for family, columns in available_measurements.items():
    region_family_presence[f"has__{family}"] = (
        region_family_presence[columns].notna().any(axis=1) if columns else False
    )

availability_rows = []
for process in PROCESS_REGISTRY:
    required = process["required"]
    supporting = process["supporting"]
    required_global = [f for f in required if family_available(f)]
    missing_global = [f for f in required if not family_available(f)]
    flags = [f"has__{f}" for f in required]
    compatible = region_family_presence[flags].all(axis=1) if flags else pd.Series(False, index=regions_df.index)
    partial = region_family_presence[flags].any(axis=1) if flags else pd.Series(False, index=regions_df.index)
    availability_rows.append({
        "process_id": process["process_id"], "kind": process["kind"],
        "required_count": len(required), "required_available_globally": len(required_global),
        "missing_global_families": ", ".join(missing_global),
        "co_located_region_count": int(compatible.sum()),
        "partial_screen_region_count": int(partial.sum()),
        "co_located_region_fraction": float(compatible.mean()),
        "supporting_available_globally": ", ".join(f for f in supporting if family_available(f)),
        "status": "screenable_with_compatible_regions" if compatible.any() else ("partial_screen_only" if partial.any() else "missing_required_data"),
    })
evidence_availability_df = pd.DataFrame(availability_rows).sort_values(["co_located_region_count", "process_id"], ascending=[False, True])
display(evidence_availability_df)
print("Evidence status is region-compatible screening availability, not process validation.")


## Candidate regions and distributions

Reviewable anomalies and rare broad types are prioritized. The priority score is only a follow-up ordering based on unusualness, cluster confidence, and rarity; it is **not** a process probability.


In [ ]:
regions_work = regions_df.copy()
for column in ["anomaly_score", "cluster_confidence"]:
    if column not in regions_work:
        regions_work[column] = np.nan
    regions_work[column] = pd.to_numeric(regions_work[column], errors="coerce")
rare = regions_work["rare_broad_type"].fillna(False).astype(bool) if "rare_broad_type" in regions_work else pd.Series(False, index=regions_work.index)
reviewable = regions_work["is_reviewable_anomaly"].fillna(False).astype(bool) if "is_reviewable_anomaly" in regions_work else pd.Series(False, index=regions_work.index)
regions_work["rare_or_reviewable"] = rare | reviewable
regions_work["anomaly_rank"] = regions_work["anomaly_score"].rank(pct=True).fillna(0.0)
regions_work["confidence_rank"] = regions_work["cluster_confidence"].rank(pct=True).fillna(0.0)
regions_work["follow_up_priority"] = 0.55 * regions_work["anomaly_rank"] + 0.25 * regions_work["confidence_rank"] + 0.20 * regions_work["rare_or_reviewable"].astype(float)
candidate_regions_df = regions_work.loc[regions_work["rare_or_reviewable"]].sort_values("follow_up_priority", ascending=False)
candidate_columns = [c for c in ["region_id","scale","analysis_cohort","neutral_type","centroid_latitude","centroid_longitude","cell_count","area_km2","cluster_confidence","anomaly_score","rare_broad_type","is_reviewable_anomaly","follow_up_priority"] if c in candidate_regions_df]
display(candidate_regions_df[candidate_columns].head(30))

group_columns = [c for c in ["scale", "analysis_cohort", "neutral_type"] if c in regions_work]
numeric_columns = [c for c in regions_work.select_dtypes(include=np.number).columns if c not in {"row", "column", "region_numeric_id"} and not c.endswith("_rank")]
distribution_rows = []
for keys, group in regions_work.groupby(group_columns, observed=True, dropna=False):
    if not isinstance(keys, tuple): keys = (keys,)
    row = dict(zip(group_columns, keys)); row["region_count"] = len(group)
    for feature in numeric_columns:
        values = pd.to_numeric(group[feature], errors="coerce").dropna()
        if len(values):
            row[f"{feature}__q10"] = float(values.quantile(0.10)); row[f"{feature}__q50"] = float(values.quantile(0.50)); row[f"{feature}__q90"] = float(values.quantile(0.90))
    distribution_rows.append(row)
neutral_type_measurement_distributions_df = pd.DataFrame(distribution_rows)
print(f"Candidate regions: {len(candidate_regions_df):,}")
print(f"Neutral-type distribution rows: {len(neutral_type_measurement_distributions_df):,}")


In [ ]:
# Keep only process-region pairs with process-specific evidence in that region.
queue_rows = []
for process in PROCESS_REGISTRY:
    required = process["required"]
    for region_index, region in candidate_regions_df.head(100).iterrows():
        present = [f for f in required if bool(region_family_presence.loc[region_index, f"has__{f}"])]
        complete = len(present) == len(required)
        if not complete:
            continue
        queue_rows.append({
            "process_id": process["process_id"], "region_id": region["region_id"],
            "neutral_type": region.get("neutral_type"), "scale": region.get("scale"),
            "analysis_cohort": region.get("analysis_cohort"),
            "centroid_latitude": region.get("centroid_latitude", np.nan),
            "centroid_longitude": region.get("centroid_longitude", np.nan),
            "follow_up_priority": float(region.get("follow_up_priority", 0.0)),
            "required_families": ", ".join(required),
            "present_required_families": ", ".join(present),
            "missing_required_families": ", ".join(f for f in required if f not in present),
            "screening_score": len(present) / max(1, len(required)),
            "status": "screenable_but_unvalidated",
            "interpretation": "No process label assigned; inspect required evidence and alternatives.",
        })
process_review_queue_df = pd.DataFrame(queue_rows)
if process_review_queue_df.empty:
    process_review_queue_df = pd.DataFrame(columns=["process_id", "region_id", "screening_score", "status"])
display(process_review_queue_df.head(30))
print(f"Process-specific review rows: {len(process_review_queue_df):,} (all required screening families are co-located)")


## Visual checks

The first plot shows why candidates remain unvalidated: required evidence families are missing from the global products. The second plot shows where unusual/rare regions are located. These are prioritization views, not geological maps.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
plot_df = evidence_availability_df.sort_values("co_located_region_fraction")
axes[0].barh(plot_df["process_id"], plot_df["co_located_region_fraction"], color="#2563EB")
axes[0].set_xlim(0, 1.05); axes[0].set_xlabel("Fraction of regions with all required screening families")
axes[0].set_title("Evidence availability — not process confidence", loc="left", fontweight="bold")
if len(candidate_regions_df):
    points = axes[1].scatter(candidate_regions_df["centroid_longitude"], candidate_regions_df["centroid_latitude"], c=candidate_regions_df["follow_up_priority"], cmap="magma", s=18, alpha=0.85)
    axes[1].set_title("Mars candidate review locations", loc="left", fontweight="bold"); axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
    axes[1].set_xlim(-180, 180); axes[1].set_ylim(-90, 90); plt.colorbar(points, ax=axes[1], label="follow-up priority")
else:
    axes[1].text(0.5, 0.5, "No reviewable candidates", ha="center", va="center"); axes[1].set_axis_off()
plt.show()


In [ ]:
control_rows = []
if not reference_locations_df.empty and {"latitude", "longitude"}.issubset(reference_locations_df.columns):
    geo = regions_work.loc[regions_work[["centroid_latitude", "centroid_longitude"]].notna().all(axis=1)]
    for control in reference_locations_df.itertuples(index=False):
        distance = (geo["centroid_latitude"] - float(control.latitude)) ** 2 + (geo["centroid_longitude"] - float(control.longitude)) ** 2
        nearest = geo.loc[distance.idxmin()] if len(distance) else None
        control_rows.append({"name": getattr(control, "name", "unknown"), "latitude": float(control.latitude), "longitude": float(control.longitude), "nearest_region_id": nearest["region_id"] if nearest is not None else None, "nearest_neutral_type": nearest["neutral_type"] if nearest is not None else None, "validation_status": "control_requires_targeted_evidence"})
control_review_df = pd.DataFrame(control_rows)
display(control_review_df)


In [ ]:
def json_safe(value: Any) -> Any:
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)): return bool(value)
    if isinstance(value, Path): return str(value)
    raise TypeError(type(value).__name__)

literature_registry_df.to_csv(OUTPUT_ROOT / "mars_literature_process_registry.csv", index=False)
evidence_availability_df.to_csv(OUTPUT_ROOT / "mars_process_evidence_availability.csv", index=False)
candidate_regions_df[candidate_columns].to_parquet(OUTPUT_ROOT / "mars_candidate_regions.parquet", index=False)
process_review_queue_df.to_parquet(OUTPUT_ROOT / "mars_process_review_queue.parquet", index=False)
neutral_type_measurement_distributions_df.to_parquet(OUTPUT_ROOT / "mars_neutral_type_measurement_distributions.parquet", index=False)
control_review_df.to_csv(OUTPUT_ROOT / "mars_control_review.csv", index=False)
summary = {"status":"candidate_review_queue_created", "input_root":str(INPUT_ROOT), "region_count":int(len(regions_df)), "neutral_type_count":int(regions_df["neutral_type"].nunique(dropna=True)) if "neutral_type" in regions_df else 0, "literature_process_count":int(len(literature_registry_df)), "candidate_region_count":int(len(candidate_regions_df)), "review_queue_rows":int(len(process_review_queue_df)), "processes_with_compatible_screening_regions":int((evidence_availability_df["co_located_region_count"] > 0).sum()), "validated_process_count":0, "earth_matching_ready":False, "scientific_warning":"No process label is valid until required observations, alternatives, positive controls, and negative controls pass."}
(OUTPUT_ROOT / "run_summary.json").write_text(json.dumps(summary, indent=2, default=json_safe), encoding="utf-8")
summary_markdown = f'''# Mars process-validation summary

- Input regions: **{summary['region_count']:,}**
- Neutral types: **{summary['neutral_type_count']:,}**
- Literature-backed process hypotheses: **{summary['literature_process_count']}**
- Candidate regions queued for review: **{summary['candidate_region_count']:,}**
- Process-specific review rows: **{summary['review_queue_rows']:,}**
- Processes with compatible screening regions: **{summary['processes_with_compatible_screening_regions']}**
- Independently validated processes: **0**
- Earth matching ready: **False**

The notebook creates evidence requirements and a targeted review queue. It does not convert neutral clusters into process labels.
'''
(OUTPUT_ROOT / "run_summary.md").write_text(summary_markdown, encoding="utf-8")
display(Markdown(summary_markdown))
